# RI-JK + DFT RKS Hessian 分解概览 (B3LYP)

本文档对标 `06-1-decomp_nh3_r_tpss0.ipynb`。B3LYP 是 hybrid GGA 泛函，其 Hessian 分解结构与 TPSS0 完全相同，差别在于：

1. hybrid 系数 $c_K = 0.20$（B3LYP）。
2. xc_type 是 GGA，不含 tau；MGGA 相关分支退化。
3. B3LYP 不含 NLC（VV10）。


In [1]:
from pyscf import gto, dft, lib
import numpy as np
from pyscf.hessian import rhf as rhf_hess
from pyscf.hessian import rks as rks_hess
from pyscf.df.hessian import rhf as df_rhf_hess
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
mf = dft.RKS(mol, xc="B3LYP").density_fit()
dat0 = np.load("nh3_r_b3lyp.npz")
mf.mo_coeff = dat0["mo_coeff"]
mf.mo_occ = dat0["mo_occ"]
mf.mo_energy = dat0["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mf_hess = mf.Hessian()
mf_hess.auxbasis_response = 2
de_ref = mf_hess.kernel().copy()
print("de_ref shape:", de_ref.shape)

de_ref shape: (4, 4, 3, 3)


## Hybrid 系数与 XC 类型确认

B3LYP 是 hybrid GGA，hybrid 系数 $c_K = 0.20$，且无 range separation、无 NLC。


In [5]:
ni = mf._numint
omega, alpha, hyb = ni.rsh_and_hybrid_coeff(mf.xc, spin=mol.spin)
print(f"omega={omega}, alpha={alpha}, hyb={hyb}")
print("is_hybrid_xc:", ni.libxc.is_hybrid_xc(mf.xc))
print("xc_type:", ni._xc_type(mf.xc))
print("do_nlc:", mf.do_nlc())
assert omega == 0.0  # no range separation
assert not mf.do_nlc()

omega=0.0, alpha=0.2, hyb=0.2
is_hybrid_xc: True
xc_type: GGA
do_nlc: 0


## Hessian 分解概览

对于 B3LYP 的 Hessian，其大致分为：

1. **密度矩阵非依赖项**：原子核排斥能的导数 `de_nuc`。与 RHF 完全相同。
2. **密度矩阵一阶项导数**：`de_1`（hcore + ovlp 部分）。与 RHF 完全相同。
3. **J/K 复杂 skeleton 导数**：`de_J20/J11/J02` 和 `de_K20/K11/K02`。K 的最终系数是 $-\tfrac{1}{2} c_K = -0.10$。
4. **XC 数值积分 skeleton 导数**：`de_vxc`。由 `rks_hess._get_vxc_diag` 与 `rks_hess._get_vxc_deriv2` 组装；这里 GGA 分支会被自动选择。
5. **CP-KS 贡献**：`de_cphf`。


In [6]:
# 1. Density matrix independent term
de_nuc = rhf_hess.hess_nuc(mol)

In [7]:
# 2 + 3. Hcore / overlap skeleton; same shape as RHF.
# auxbasis_response = 0: 只含 orbital derivative
hessobj_aux0 = mf.Hessian()
hessobj_aux0.auxbasis_response = 0
de_1, ej_aux0, ek_aux0 = df_rhf_hess._partial_hess_ejk(hessobj_aux0, with_k=True)

In [8]:
# auxbasis_response = 1: 1st-order aux response (Hessian 贡献会被缩放 0.5)
hessobj_aux1 = mf.Hessian()
hessobj_aux1.auxbasis_response = 1
_, ej_aux1, ek_aux1 = df_rhf_hess._partial_hess_ejk(hessobj_aux1, with_k=True)

In [9]:
# auxbasis_response = 2: full aux response
hessobj_aux2 = mf.Hessian()
hessobj_aux2.auxbasis_response = 2
_, ej_aux2, ek_aux2 = df_rhf_hess._partial_hess_ejk(hessobj_aux2, with_k=True)

In [10]:
# 4-a. J/K skeleton decomposition (identical to RHF; K is later multiplied by hyb)
de_J20 = ej_aux0.copy()
de_J11 = 2.0 * (ej_aux1 - ej_aux0)
de_J02 = ej_aux2 - 2.0 * ej_aux1 + ej_aux0

de_K20 = 2 * ek_aux0.copy()
de_K11 = 2 * 2.0 * (ek_aux1 - ek_aux0)
de_K02 = 2 * (ek_aux2 - 2.0 * ek_aux1 + ek_aux0)

### 4-b. XC 数值积分 skeleton 二阶导数

与 TPSS0 一致，调用 `rks_hess._get_vxc_diag` 与 `rks_hess._get_vxc_deriv2`。GGA 分支只涉及 RHO 与 SIGMA 部分，不含 tau。这里仍不考虑格点权重的导数。


In [11]:
# 4-b. XC numerical-integration skeleton (no grid response).
max_memory = 4000
veff_diag = rks_hess._get_vxc_diag(hessobj_aux0, mf.mo_coeff, mf.mo_occ, max_memory)
vxc_list = rks_hess._get_vxc_deriv2(hessobj_aux0, mf.mo_coeff, mf.mo_occ, max_memory)

mocc = mf.mo_coeff[:, mf.mo_occ > 0]
dm0 = mocc @ mocc.T * 2
aoslices = mol.aoslice_by_atom()
natm = mol.natm
de_vxc = np.zeros((natm, natm, 3, 3))
for A in range(natm):
    p0, p1 = aoslices[A][2:]
    de_vxc[A, A] += np.einsum("xypq,pq->xy", veff_diag[:, :, p0:p1], dm0[p0:p1]) * 2
    veff_A = vxc_list[A]
    for B in range(A + 1):
        q0, q1 = aoslices[B][2:]
        de_vxc[A, B] += np.einsum("xypq,pq->xy", veff_A[:, :, q0:q1], dm0[q0:q1]) * 2
    for B in range(A):
        de_vxc[B, A] = de_vxc[A, B].T

## CP-KS 响应

记录电子部分总贡献后，再减去已经分解出的电子项，剩下的就是 CP-KS 贡献。


In [12]:
# 5. CP-KS contribution
de_hess_elec = mf_hess.hess_elec()
de_partial = de_1 + ej_aux2 - hyb * ek_aux2 + de_vxc
de_cphf = de_hess_elec - de_partial

## 总核验

总和形式为

$$
\mathrm{d}^2 E = \underbrace{\texttt{de\_1}}_{\text{hcore+ovlp}}
+ \underbrace{\texttt{de\_J20+de\_J11+de\_J02}}_{\text{J skeleton}}
- \tfrac{1}{2} c_K \underbrace{(\texttt{de\_K20+de\_K11+de\_K02})}_{\text{K skeleton}}
+ \underbrace{\texttt{de\_vxc}}_{\text{XC skeleton}}
+ \underbrace{\texttt{de\_cphf}}_{\text{CP-KS}}
+ \underbrace{\texttt{de\_nuc}}_{\text{nuc-nuc}}
$$

其中 $c_K = 0.20$ 是 B3LYP 的 hybrid 系数。


In [13]:
de_sum = (
    de_1
    + de_J20 + de_J11 + de_J02
    - 0.5 * hyb * (de_K20 + de_K11 + de_K02)
    + de_vxc
    + de_cphf
    + de_nuc
)
print("de_ref == de_sum:", np.allclose(de_ref, de_sum))
print("max abs difference:", np.max(np.abs(de_ref - de_sum)))

de_ref == de_sum: True
max abs difference: 4.711786516509164e-13


最终，我们将这些分量都放到 `nh3_r_b3lyp_decomp.npz` 文件中，用于后续的核验和分析。


In [14]:
dat = dict(np.load("nh3_r_b3lyp.npz"))
dat.update({
    "de_nuc": de_nuc,
    "de_1": de_1,
    "de_J20": de_J20,
    "de_J11": de_J11,
    "de_J02": de_J02,
    "de_K20": de_K20,
    "de_K11": de_K11,
    "de_K02": de_K02,
    "de_vxc": de_vxc,
    "de_cphf": de_cphf,
    "de_ref": de_ref,
    "hyb": np.asarray(hyb),
})
np.savez("nh3_r_b3lyp_decomp.npz", **dat)